# Google Agent Development Kit (ADK) — Complete Agentic AI & Multi-Agent Masterclass
### LangGraph curriculum → Google ADK 2.x equivalents + extra production capabilities

**Research verified on 10 October 2026** against [Google ADK documentation](https://adk.dev/) and [Google ADK GitHub](https://github.com/google/adk-python). This notebook targets **Python `google-adk==2.10.0`** (September 2026 stable release); Python **3.10+**.

**Audience:** AI engineers, Python developers, startups/GCC/MNC teams, and advanced bootcamp learners.

**Learning outcome:** From an agent with tools to stateful graphs, persisted conversations, reusable subgraphs, human approvals, multi-agent systems, and a complete **IT Incident Resolution Agent** capstone. Further chapters cover A2A, MCP, skills, Google Search grounding, multimodal, artifacts, streaming, tracing, evaluation, Cloud Run and managed runtime.

> **Safety and cost:** `RUN_LIVE = False` by default. No Gemini API calls will run until you explicitly opt in. All incident investigation/remediation uses **mock/simulated data only**; never run destructive operational commands based solely on model suggestions.

> **Validation:** Notebook JSON and Python cell syntax are checked. Examples follow official ADK 2.x docs. This authoring environment cannot download ADK or execute Gemini requests, so **full ADK runtime/API executions are not claimed as tested**. Run the environment verification cells in your machine before a live class.

## 0. Map of your LangGraph classes (41–50) to Google ADK

| LangGraph topic from your course | Google ADK API/pattern | Nature of equivalence |
|---|---|---|
| Intro, graphs, agents | `LlmAgent`, `Runner`, **ADK 2.x `Workflow`** | Native agent + graph workflows |
| Tool calling, ReAct agent | Python functions in `tools`, automatic model/tool loop | Native |
| Pydantic / structured output | `output_schema=BaseModel`, `input_schema`, `output_key` | Native; model compatibility matters |
| Typed graph state | `Session.state`, event/state deltas, typed graph node data | Different state model; not LangGraph reducers |
| Memory management | `SessionService`, `MemoryService`, context compression | Native + custom stores |
| SQLite checkpoints | `DatabaseSessionService` via async SQLite URL | **Session persistence**, not exact LangGraph checkpoint semantics |
| Long-term memory / LangMem | `InMemoryMemoryService`, `load_memory`, external memory bank | Native memory services; not LangMem itself |
| Human-in-the-loop | `ToolContext.request_confirmation` or graph `RequestInput` | Native, different mechanisms |
| Conditional routing / Command | `Workflow` `Event(route=...)`, agent handoff | Native routing |
| Subgraphs | Nested `Workflow` and reusable `SequentialAgent` | Native composition |
| Parallel fan-out/fan-in / Send | `ParallelAgent` or `Workflow` + `JoinNode` | Native |
| Workflow patterns | `SequentialAgent`, `ParallelAgent`, `LoopAgent`, `Workflow` | Native |
| Complete multi-agent flow | Specialized `LlmAgent`s, coordinator, workflow agents | Native |
| State rewind / time travel | `Runner.rewind_async` for supported session rewind | Partial conceptual overlap |

### ADK-only/extra material
Tools + callbacks + plugins, artifact service, A2A remote agents, MCP integration, grounding with Google Search, multimodal/live sessions, evaluation, deployment, observability, and experimental agent skills.

**Critical 2026 change:** Google ADK 2.x includes **first-class graphs** (`Workflow`, nodes, routes, `JoinNode`); don't assume it only supports 1.x `SequentialAgent`/`ParallelAgent`/`LoopAgent`. [Official graph routes](https://adk.dev/graphs/routes/) · [2.10.0 release](https://github.com/google/adk-python/releases/tag/v2.10.0).

## 1. Setup — pip / uv / secrets / run switches

Recommended create environment (terminal):

```bash
uv venv .venv --python 3.11
# activate .venv (Windows: .venv\Scripts\activate | Mac/Linux: source .venv/bin/activate)
uv pip install "google-adk==2.10.0" "pydantic>=2" python-dotenv ipykernel
python -m ipykernel install --user --name google-adk-masterclass
```

Alternatively run the following installation cell in this notebook, **restart kernel**, then run imports. Model: choose an available Gemini API model, default `gemini-2.5-flash` (you can override). Set key in environment or a local ignored `.env`: `GOOGLE_API_KEY=...`. NEVER place a real key into a notebook and commit it.

In [ ]:
# Execute once; restart Jupyter kernel if it installs or upgrades the library.
%pip install "google-adk==2.10.0" "pydantic>=2" python-dotenv ipykernel

In [ ]:
import os, json, sqlite3, asyncio, uuid, time
from pathlib import Path
from pprint import pprint
from typing import Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from google.genai import types
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, DatabaseSessionService

load_dotenv()
RUN_LIVE = False        # Deliberately off: requires a Gemini API key and may incur cost
RUN_GRAPH_DEMOS = False   # For graph runtime execution on your installed ADK 2.x
MODEL = os.getenv("GOOGLE_ADK_MODEL", "gemini-2.5-flash")
APP = "google_adk_bootcamp"
USER = "classroom_student"
print("Using model:", MODEL)
print("Google API key present:", bool(os.getenv("GOOGLE_API_KEY")))
try:
    import importlib.metadata
    print("Installed google-adk:", importlib.metadata.version("google-adk"))
except Exception as exc:
    print("Could not read package version:", exc)

## 2. Runner lifecycle, messages, events and reusable notebook helper

`LlmAgent` defines the agent; `Runner` executes it using a `SessionService`. Sessions are keyed by `app_name`, `user_id`, and `session_id`. A runner yields **events**, not simply a response string. `event.is_final_response()` identifies the final response when present. The helper creates a fresh session and gives you both events and final text.

Official: [Getting started](https://adk.dev/get-started/python/) · [Sessions](https://adk.dev/sessions/session/).

In [ ]:
def user_text(message: str) -> types.Content:
    return types.Content(role="user", parts=[types.Part(text=message)])

async def run_agent(agent, prompt: str, *, app_name: str | None = None,
                    service=None, user_id: str = USER, session_id: str | None = None):
    app_name = app_name or f"app_{agent.name}"
    service = service or InMemorySessionService()
    session_id = session_id or uuid.uuid4().hex
    existing = await service.get_session(
        app_name=app_name, user_id=user_id, session_id=session_id
    )
    if existing is None:
        await service.create_session(
            app_name=app_name, user_id=user_id, session_id=session_id
        )
    runner = Runner(agent=agent, app_name=app_name, session_service=service)
    events, final_texts = [], []
    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=user_text(prompt)
    ):
        events.append(event)
        if event.is_final_response() and event.content:
            final_texts.extend(
                part.text for part in (event.content.parts or []) if part.text
            )
    return {"text": "\n".join(final_texts), "events": events,
            "session_id": session_id, "service": service, "runner": runner}

print("Reusable Runner helper ready. Live calls are OFF by default.")

## 3. First Google ADK agent (`LlmAgent`)

In [ ]:
hello_agent = LlmAgent(
    name="hello_teacher",
    model=MODEL,
    description="Explains technical concepts to new engineers.",
    instruction="Teach Python and agentic AI clearly. Use concise examples."
)
if RUN_LIVE:
    reply = await run_agent(hello_agent, "What is an AI agent in two examples?")
    print(reply["text"])
else:
    print("Constructed:", hello_agent.name, "(enable RUN_LIVE for Gemini output)")

## 4. Create custom tools and the automatic ReAct-style agent/tool loop

Plain, documented Python functions are valid ADK tools. The model can choose to call them; ADK handles the function-call result and another model turn. **A function tool is not the same as an agent; the agent selects it dynamically.** [Function tools docs](https://adk.dev/tools-custom/function-tools/).

In [ ]:
INVENTORY = {"catalog": {"healthy": True, "latency_ms": 35},
             "payment": {"healthy": False, "latency_ms": 960}}

def get_service_health(service: str) -> dict:
    """Return simulated service health. Args: service: catalog or payment."""
    record = INVENTORY.get(service.lower())
    return {"service": service, "found": bool(record), "metrics": record}

def get_runbook(service: str) -> dict:
    """Fetch a safe read-only runbook for a simulated service."""
    return {"service": service, "steps": ["Inspect metrics", "Check logs",
                                           "Review deploy history", "Seek approval"]}

print(get_service_health("payment"))             # OFFLINE executable
print(get_runbook("payment"))                    # OFFLINE executable

tools_agent = LlmAgent(
    name="operations_assistant", model=MODEL,
    instruction="Investigate services using the tools. Never claim to have repaired anything.",
    tools=[get_service_health, get_runbook]
)
if RUN_LIVE:
    print((await run_agent(tools_agent, "Investigate payment service health"))["text"])

## 5. Pydantic structured outputs (`output_schema`) + `output_key`

`output_schema` enforces structured **final output**, and `output_key` writes that output to session state. Official ADK docs caution that the combination of `output_schema` and function tools in the **same LLM request** depends on model support. Use a separate formatter agent if your model does not support this combination. [Official structured outputs](https://adk.dev/agents/llm-agents/).

In [ ]:
class IncidentAssessment(BaseModel):
    service: str = Field(description="Impacted service")
    severity: Literal["low", "medium", "high", "critical"]
    hypothesis: str
    actions: list[str]

structured_agent = LlmAgent(
    name="incident_classifier", model=MODEL,
    instruction="Analyze the incident and return the required structured response.",
    output_schema=IncidentAssessment, output_key="incident_assessment"
)
assert IncidentAssessment(service="payment", severity="high",
                          hypothesis="elevated latency", actions=["Inspect logs"])
if RUN_LIVE:
    res = await run_agent(structured_agent, "Payments latency is 960ms, 28% errors.")
    print(res["text"])
    sess = await res["service"].get_session(app_name="app_incident_classifier",
                user_id=USER,session_id=res["session_id"])
    print("Stored state:",sess.state.get("incident_assessment"))

## 6. Agent state, `Session.state`, state namespaces, and `ToolContext`

ADK state comes from event-backed session data; you can access and modify it within tool context. Common namespaces: unprefixed session-scoped keys, `user:` user state, `app:` shared application state, `temp:` short-lived values. **Do not use `Session.state` as an authorization database**; scope access via trusted backend identity.

In [ ]:
from google.adk.tools import ToolContext

def update_priority(priority: str, tool_context: ToolContext) -> dict:
    """Save an incident priority to ADK session state."""
    allowed = {"low", "medium", "high", "critical"}
    if priority.lower() not in allowed:
        return {"error": "invalid priority"}
    tool_context.state["incident_priority"] = priority.lower()
    return {"saved": True, "priority": priority.lower()}

state_agent = LlmAgent(name="stateful_triage", model=MODEL,
    instruction="Use update_priority to store priority when asked.",
    tools=[update_priority])
print("State tool registered:", state_agent.name)
if RUN_LIVE:
    print((await run_agent(state_agent,"Set incident priority high"))["text"])

## 7. Multi-turn short-term memory with the same session

Session histories are not model weights. You must reuse **the same app, user, session ID** to maintain conversation continuity. A new session starts a separate conversation.

In [ ]:
chat_service = InMemorySessionService()
chat_app, chat_sid = "adk_chat_memory", "class_demo_001"
chat_agent = LlmAgent(name="chat_memory_tutor", model=MODEL,
                      instruction="Be helpful and use relevant conversation history.")
if RUN_LIVE:
    a = await run_agent(chat_agent, "My project name is Phoenix.",
                       app_name=chat_app, service=chat_service,session_id=chat_sid)
    b = await run_agent(chat_agent, "What is the name of my project?",
                       app_name=chat_app, service=chat_service,session_id=chat_sid)
    print("Turn 2:", b["text"])
else:
    print("Same session must be used for turns 1 and 2:", chat_sid)

## 8. Persistent SQLite sessions (similar *purpose*, different checkpoint semantics)

`DatabaseSessionService(db_url='sqlite+aiosqlite:///./file.db')` persists ADK sessions. It is **not** a drop-in LangGraph graph checkpointer with identical resume/fork guarantees. Production: use proper SQL credentials/migrations, retention and access controls.

In [ ]:
persistent_service = DatabaseSessionService(
    db_url="sqlite+aiosqlite:///./google_adk_sessions.db"
)
sqlite_agent = LlmAgent(name="sqlite_memory_assistant",model=MODEL,
                       instruction="Help users with operational questions.")
if RUN_LIVE:
    res = await run_agent(sqlite_agent, "Remember that our on-call group is Delta.",
        app_name="persistent_sessions", service=persistent_service,
        session_id="training_persistent_1")
    print(res["text"])
else:
    print("Configured persistent ADK sessions with SQLite.")

## 9. ADK MemoryService — *longer-term searchable context*

MemoryService is distinct from SessionService. Example with `InMemoryMemoryService` (educational; not durable). After a session, ingest it via `add_session_to_memory`. Add the `load_memory` or `preload_memory` tool to enable model lookup. Enterprise: consider supported persistent/managed memory providers or an approved external database.

Official: [ADK Memory](https://adk.dev/sessions/memory/).

In [ ]:
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory

memory_service = InMemoryMemoryService()
memory_agent = LlmAgent(name="memory_specialist", model=MODEL,
    instruction="Use load_memory to consult relevant remembered conversations.",
    tools=[load_memory])
mem_sessions = InMemorySessionService()
if RUN_LIVE:
    await run_agent(memory_agent, "Remember incident INC-900: payment latency root cause cache.",
                    app_name="adk_memory_app", service=mem_sessions,
                    session_id="past_incident")
    saved = await mem_sessions.get_session(app_name="adk_memory_app",
                             user_id=USER,session_id="past_incident")
    await memory_service.add_session_to_memory(saved)
    runner = Runner(agent=memory_agent,app_name="adk_memory_app",
                    session_service=mem_sessions, memory_service=memory_service)
    await mem_sessions.create_session(app_name="adk_memory_app",user_id=USER,
                                      session_id="future_incident")
    async for event in runner.run_async(user_id=USER,session_id="future_incident",
                       new_message=user_text("What caused INC-900?")):
        if event.is_final_response() and event.content:
            print(" ".join(p.text or "" for p in event.content.parts))
else:
    print("Configured ADK MemoryService and a load_memory-enabled agent.")

## 10. Custom durable long-term memory — SQLite table as a learning pattern

This is intentionally ordinary Python: durable memory requires tenancy, consent, retention, deduplication, retrieval/ranking and authorization policies. The example avoids an LLM call and **must not be used for real personal/sensitive data without controls**.

In [ ]:
MEMORY_DB = Path("google_adk_teaching_longterm_memory.db")
with sqlite3.connect(MEMORY_DB) as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS memories (tenant TEXT, memory_id TEXT PRIMARY KEY, fact TEXT)")

def remember_fact(tenant: str, fact: str) -> str:
    """Store one deliberately non-sensitive mock fact for a tenant."""
    mem_id = uuid.uuid4().hex
    with sqlite3.connect(MEMORY_DB) as conn:
        conn.execute("INSERT INTO memories VALUES (?,?,?)", (tenant, mem_id, fact))
    return mem_id

def recall_facts(tenant: str, query: str) -> list[str]:
    """Trivial SQL LIKE search for teaching only; replace with proper retrieval."""
    with sqlite3.connect(MEMORY_DB) as conn:
        rows = conn.execute("SELECT fact FROM memories WHERE tenant=? AND fact LIKE ? LIMIT 5",
                            (tenant, f"%{query}%")).fetchall()
    return [r[0] for r in rows]

_demo_mem_id = remember_fact("demo_org", "Runbook payment: review recent deploy before changes")
print("Saved:", _demo_mem_id[:8], "Retrieved:", recall_facts("demo_org", "payment"))

## 11. Memory management — summarization / context compression / caching

ADK offers **context compression** and context caching options. Neither makes a persistent memory store by itself. Typical production strategy: bounded history → running summary → targeted retrieval → old-event retention policy. See [context compression](https://adk.dev/sessions/context-compression/) and [context caching](https://adk.dev/sessions/context-caching/).

In [ ]:
def compact_messages(messages: list[str], keep_latest: int = 4) -> dict:
    """Illustrative APP-LEVEL policy, not an ADK built-in summarizer."""
    return {"archived_count": max(0, len(messages) - keep_latest),
            "recent": messages[-keep_latest:]}
print(compact_messages([f"message-{i}" for i in range(8)]))

## 12. Before/after callbacks, tool-argument guardrails, policy gates

Callbacks can block/model-modify/run custom logic around agent/model/tool steps. Here a **read-only tool** access policy prevents unsupported service names. Returning a `dict` from a `before_tool_callback` skips tool execution with that simulated result. [Official callbacks](https://adk.dev/callbacks/).

In [ ]:
ALLOWED_SERVICES = {"catalog", "payment"}

def restrict_service_tool(tool, args: dict, tool_context):
    service = args.get("service")
    if service is not None and service not in ALLOWED_SERVICES:
        return {"error": "Blocked by allowlist", "service": service}
    return None   # proceed

safe_tools_agent = LlmAgent(name="guarded_operations",model=MODEL,
    instruction="Look up permitted service health only.",
    tools=[get_service_health], before_tool_callback=restrict_service_tool)
print("Guardrail test:",restrict_service_tool(None,{"service":"unknown"},None))
if RUN_LIVE:
    print((await run_agent(safe_tools_agent,"Check an unknown service"))["text"])

## 13. Multi-agent delegation with `sub_agents` (LLM-selected routing)

A coordinator can delegate to child agents based on their **name**, **description**, and instructions. This differs from deterministic workflow execution. In production, make handoff authority and boundaries explicit.

In [ ]:
billing_child = LlmAgent(name="billing_specialist",model=MODEL,
    description="Investigates invoice and payment questions.",
    instruction="Answer only billing/invoice questions.")
tech_child = LlmAgent(name="technical_specialist",model=MODEL,
    description="Investigates API availability, incidents and latency.",
    instruction="Answer only technical infrastructure questions.")
coordinator = LlmAgent(name="service_coordinator",model=MODEL,
    description="Routes requests to specialist agents.",
    instruction="Delegate billing issues to billing_specialist and technical issues to technical_specialist.",
    sub_agents=[billing_child, tech_child])
if RUN_LIVE:
    print((await run_agent(coordinator,"Payment service throws HTTP 502"))["text"])

## 14. Agents-as-tools vs handoff (manager keeps control)

Use `AgentTool` when a supervisor wants a specialist's result returned as a tool output, instead of delegating the ongoing conversation. This is different from `sub_agents` routing and is useful for report assembly.

In [ ]:
from google.adk.tools.agent_tool import AgentTool

research_worker = LlmAgent(name="research_worker",model=MODEL,
    description="Summarizes technical evidence.",
    instruction="Return 3 evidence bullets only.")
manager_agent = LlmAgent(name="research_manager",model=MODEL,
    instruction="Use the research_worker tool, then deliver a clear final summary.",
    tools=[AgentTool(agent=research_worker)])
if RUN_LIVE:
    print((await run_agent(manager_agent,"Why is our mock payment service slow?"))["text"])

## 15. Deterministic SequentialAgent: plan → draft → review

Unlike routing with `sub_agents`, **every** `SequentialAgent` child runs in order. Use `output_key` to pass a child output to later instructions (e.g. `{draft_text}`).

In [ ]:
draft_agent = LlmAgent(name="draft_generator",model=MODEL,
    instruction="Draft a 3-line incident update. Avoid made-up claims.",
    output_key="draft_text")
review_agent = LlmAgent(name="draft_reviewer",model=MODEL,
    instruction="Improve this draft for clarity: {draft_text}. Return final bulletin.",
    output_key="reviewed_text")
sequential_pipeline = SequentialAgent(name="sequential_bulletin",
    sub_agents=[draft_agent,review_agent])
if RUN_LIVE:
    print((await run_agent(sequential_pipeline,"Payment latency is elevated"))["text"])

## 16. ParallelAgent: parallel specialist investigations

In a `ParallelAgent`, child agents run concurrently with their own execution branches. To aggregate results, combine the parallel stage with a following `SequentialAgent` summarizer. Avoid writing colliding state keys across branches.

In [ ]:
metrics_worker = LlmAgent(name="metrics_worker",model=MODEL,
    instruction="Write a 2-line simulated metrics analysis. No real monitoring access.",
    output_key="metrics_summary")
logs_worker = LlmAgent(name="logs_worker",model=MODEL,
    instruction="Write a 2-line simulated logs analysis. No real log access.",
    output_key="logs_summary")
parallel_investigations = ParallelAgent(name="parallel_investigations",
    sub_agents=[metrics_worker, logs_worker])
parallel_synth = LlmAgent(name="parallel_synthesizer",model=MODEL,
    instruction="Combine metrics: {metrics_summary} and logs: {logs_summary}. State uncertainty.")
parallel_to_summary = SequentialAgent(name="parallel_to_summary",
    sub_agents=[parallel_investigations,parallel_synth])
if RUN_LIVE:
    print((await run_agent(parallel_to_summary,"Investigate our mock incident"))["text"])

## 17. LoopAgent: iterative critic–optimizer with bounded iterations

`LoopAgent(max_iterations=...)` provides an iteration cap. A reviewer may trigger an early exit with `tool_context.actions.escalate=True` via a function tool, but the **hard cap is mandatory**. Evaluator/optimizer loops are useful for drafting and tests.

In [ ]:
def approve_draft(tool_context: ToolContext) -> dict:
    """Signal success and end a LoopAgent if the model decides the draft is acceptable."""
    tool_context.actions.escalate = True
    return {"approved": True}

loop_writer = LlmAgent(name="loop_writer",model=MODEL,
    instruction="Create or improve a short incident bulletin. Use previous feedback if provided.",
    output_key="draft")
loop_critic = LlmAgent(name="loop_critic",model=MODEL,
    instruction="Review {draft}; if adequate, call approve_draft; otherwise give concrete feedback.",
    tools=[approve_draft],output_key="feedback")
review_loop = LoopAgent(name="iterative_review",max_iterations=3,
                        sub_agents=[loop_writer,loop_critic])
if RUN_LIVE:
    print((await run_agent(review_loop,"Write a 2-sentence incident update."))["text"])

# Part II. Native ADK 2.x Graph Workflows

These are **real ADK 2.x graph primitives**, not LangGraph APIs recreated manually. Graph functions return `Event(output=...)` or `Event(route=...)`, and `Workflow(edges=[...])` defines graph connections. The runtime needs `google-adk>=2.0`; pinned `2.10.0` is used here. Set `RUN_GRAPH_DEMOS=True` separately if you want to execute deterministic workflow demos. 

Official: [Graph routes](https://adk.dev/graphs/routes/) · [Graph data handling](https://adk.dev/graphs/data-handling/) · [Dynamic workflows](https://adk.dev/graphs/dynamic/).

## 18. Graph nodes, START, edges, typed outputs — direct LangGraph-style architecture

This first graph only transforms local strings—**no model needed**. A node can be a Python function, an agent, or a human-input step.

In [ ]:
from google.adk import Workflow, Event

def normalize_issue(text: str):
    return Event(output=text.strip().lower())

def prepare_ticket(text: str):
    return Event(output=f"TICKET: {text}")

simple_graph = Workflow(name="simple_graph",edges=[
    ("START",normalize_issue,prepare_ticket)
])
if RUN_GRAPH_DEMOS:
    r = await run_agent(simple_graph,"  PAYMENT ERROR 502  ")
    print("Graph events:",[(e.author,e.is_final_response()) for e in r["events"]])
else:
    print("Built native ADK graph with START → normalize_issue → prepare_ticket")

## 19. Conditional edges and `Event(route=...)` — ADK graph router

Unlike an LLM-selected handoff, this is deterministic routing **controlled by code**. Compare with LangGraph's conditional edge or Command-goto design.

In [ ]:
def choose_team(text: str):
    route = "PAYMENT" if "pay" in text.lower() else "GENERAL"
    return Event(route=route, output=text)

def payment_path(text: str):
    return Event(output=f"Payment support queue: {text}")

def general_path(text: str):
    return Event(output=f"General support queue: {text}")

routing_graph = Workflow(name="routing_graph",edges=[
    ("START",choose_team),
    (choose_team,{"PAYMENT":payment_path,"GENERAL":general_path})
])
if RUN_GRAPH_DEMOS:
    result = await run_agent(routing_graph,"Payment checkout failed")
    print(len(result["events"]),"events")

## 20. Graph parallel fan-out + `JoinNode` fan-in (LangGraph Send-like)

A `JoinNode` awaits upstream results and emits their combined outputs. If a branch never produces output, the join can remain blocked; make branches terminate reliably. Unlike the LangGraph dynamic `Send` API, this example uses a fixed set of graph branches.

In [ ]:
from google.adk.workflow import JoinNode

def research_errors(text: str):
    return Event(output=f"errors reviewed for: {text}")

def research_latency(text: str):
    return Event(output=f"latency reviewed for: {text}")

def merge_results(items):
    return Event(output={"kind":"combined","branches":items})

join_node = JoinNode(name="investigation_join")
fanout_graph = Workflow(name="fanout_graph",edges=[
    ("START",research_errors,join_node),
    ("START",research_latency,join_node),
    (join_node,merge_results),
])
if RUN_GRAPH_DEMOS:
    print("Graph event count:",len((await run_agent(fanout_graph,"payment outage"))["events"]))

## 21. Native nested workflow (subgraph composition)

In ADK you can nest `Workflow` inside another `Workflow` for reuse; this is closer to LangGraph subgraphs than 1.x workflow templates alone. Track how nested nodes exchange output; don't assume LangGraph reducers or `Command.PARENT` semantics.

In [ ]:
def clean_text(text: str):
    return Event(output=text.strip())
def annotate_text(text: str):
    return Event(output=f"checked::{text}")
def send_text(text: str):
    return Event(output=f"delivered::{text}")

nested_graph = Workflow(name="nested_quality_workflow",edges=[
    ("START",clean_text,annotate_text)
])
parent_graph = Workflow(name="parent_graph",edges=[
    ("START",nested_graph,send_text)
])
print("Built nested ADK Workflow")

## 22. Dynamic routing, state handoff and typed node contracts

For dynamic routing patterns, see ADK's [Dynamic workflows](https://adk.dev/graphs/dynamic/). Keep node I/O explicit: typed Pydantic data or `Event(output=...)`; use `Session.state` for shared state and avoid global mutable variables in real deployments.

In [ ]:
class IncidentNodeOutput(BaseModel):
    incident_id: str
    severity: Literal["low","medium","high","critical"]
    recommended_team: str

def classify_mock_incident(raw: str) -> IncidentNodeOutput:
    severity = "critical" if "outage" in raw.lower() else "medium"
    return IncidentNodeOutput(incident_id="INC-101",severity=severity,
        recommended_team="SRE" if severity=="critical" else "Support")

example_result = classify_mock_incident("payment outage")
print(example_result.model_dump())

## 23. Native graph human-in-the-loop using `RequestInput`

A node can yield `RequestInput(message=..., payload=..., response_schema=...)` and wait for user input. The actual reply/resumption is handled by an ADK-compatible runtime/client, not by a Python `input()` in a notebook. **This constructs the graph, but the notebook does not automate the UI reply.**

Official: [Human input graph nodes](https://adk.dev/graphs/human-input/).

In [ ]:
from google.adk.events import RequestInput

def ask_for_approval():
    yield RequestInput(message="Approve this simulated service restart? Reply yes or no.",
                       response_schema=str)

def record_decision(answer: str):
    approved = str(answer).lower().strip() == "yes"
    return Event(output={"approved":approved,
                         "effect":"simulated decision only; no service called"})

graph_hitl = Workflow(name="hitl_graph",edges=[
    ("START",ask_for_approval,record_decision)
])
print("Constructed graph HITL workflow; run/resume via ADK runtime or app client.")

## 24. Native tool approval using `ToolContext.request_confirmation`

Google ADK also has **tool confirmation** separate from graph `RequestInput`. Use it for permissioned actions. A human's authorization must be tied to the exact tool arguments, identity and expiry. The following tool only simulates a restart.

Official: [Action confirmations](https://adk.dev/tools-custom/confirmation/).

In [ ]:
def restart_service_simulated(service: str, tool_context: ToolContext) -> dict:
    """Request human confirmation before a SIMULATED restart."""
    confirmation = tool_context.tool_confirmation
    if not confirmation:
        tool_context.request_confirmation(
            hint=f"Approve simulated restart of '{service}'?"
        )
        return {"status":"approval_required","service":service}
    if not confirmation.confirmed:
        return {"status":"rejected","service":service}
    return {"status":"SIMULATED_RESTART_ONLY","service":service}

approval_agent = LlmAgent(name="approval_operator",model=MODEL,
    instruction="Propose only a simulated restart and await approval.",
    tools=[restart_service_simulated])
print("Tool confirmation configured, no real mutation possible")

## 25. Resume, rewinding sessions and durable execution

- **Resume**: enabled with `App(resumability_config=ResumabilityConfig(is_resumable=True))` for supported workflows; consult the runtime guide for actual resume calls.
- **Rewind**: `await runner.rewind_async(user_id=...,session_id=...,rewind_before_invocation_id=...)` for supported session state/history rewind.
- **IMPORTANT**: Rewinding a session does **not undo external real-world side effects** (emails sent, service changes, purchases). Use idempotent tools + approval + audit logs.

Official: [Resume agents](https://adk.dev/runtime/resume/) · [Rewind sessions](https://adk.dev/sessions/session/rewind/).

In [ ]:
from google.adk.apps import App, ResumabilityConfig

resumable_app = App(name="resumable_teaching_app",root_agent=hello_agent,
    resumability_config=ResumabilityConfig(is_resumable=True))
print("Resumability configured; requires runtime support and persisted sessions.")

# Reference-only example: invoke only when you have a valid supported session/invocation.
# await runner.rewind_async(user_id=USER, session_id="existing_id",
#                           rewind_before_invocation_id="existing_invocation_id")

## 26. Decision table: which orchestration pattern to choose?

| Need | Prefer |
|---|---|
| Model chooses specialist | Parent `LlmAgent(sub_agents=[...])` |
| Parent must stay in control | `AgentTool(agent=...)` |
| Guaranteed sequence | `SequentialAgent` |
| Parallel independent workers | `ParallelAgent` |
| Bounded evaluate→retry | `LoopAgent` |
| Explicit branches, graph paths, joins | `Workflow` + `Event(route=...)` + `JoinNode` |
| Human review inside graph | Graph `RequestInput` |
| Approval before executing a tool | `ToolContext.request_confirmation` |
| Separate remote services/teams | A2A remote agents |

# Part III. Capstone — Multi-Agent IT Incident Resolution System

**End-to-end conceptual flow**

```text
Incident ticket
    ↓
Triage LlmAgent (severity + assignment)
    ↓
ParallelAgent ── logs agent ─┐
               metrics agent ─┼─→ Synthesis LlmAgent
               runbook agent ─┘
    ↓
Remediation planner (NO actual mutations)
    ↓
Human approval tool (optional separate step)
    ↓
Post-action verification and final incident report
```

All tools access **hardcoded fake telemetry**. This is an educational production architecture, NOT an authorized remediation deployment.

## 27. Capstone A — mock tools and schemas

In [ ]:
SIMULATED_INCIDENT = {
    "id":"INC-2026-101", "service":"payment", "http_5xx_rate":0.28,
    "p95_latency_ms":960, "recent_deploy":"payment-v4.2.1",
    "logs":["connection pool saturation", "28% HTTP 502"]
}

def read_mock_metrics(service: str) -> dict:
    """Return mock metrics, never real production telemetry."""
    if service != "payment": return {"error":"No data for service"}
    return {k:SIMULATED_INCIDENT[k] for k in ("service","http_5xx_rate","p95_latency_ms")}

def read_mock_logs(service: str) -> dict:
    """Return sanitized demonstration log lines."""
    return {"service":service,"lines":SIMULATED_INCIDENT["logs"] if service=="payment" else []}

def read_mock_deploys(service: str) -> dict:
    """Return a mock recent-deployment record."""
    return {"service":service,"deploy":SIMULATED_INCIDENT["recent_deploy"]}

pprint(read_mock_metrics("payment"))
pprint(read_mock_logs("payment"))

## 28. Capstone B — triage → parallel investigations → synthesis → plan

In [ ]:
triage = LlmAgent(name="incident_triage",model=MODEL,
    instruction="Classify impact and urgency for the simulated incident; no assumptions.",
    output_key="triage")
log_researcher = LlmAgent(name="incident_log_researcher",model=MODEL,
    instruction="Use the mock logs tool and cite its actual result.",
    tools=[read_mock_logs],output_key="logs_report")
metric_researcher = LlmAgent(name="incident_metric_researcher",model=MODEL,
    instruction="Use mock metrics and report evidence in two bullets.",
    tools=[read_mock_metrics],output_key="metrics_report")
deployment_researcher = LlmAgent(name="incident_deployment_researcher",model=MODEL,
    instruction="Read simulated release history, report what you find.",
    tools=[read_mock_deploys],output_key="deploy_report")
investigation_team = ParallelAgent(name="incident_investigation_team",sub_agents=[
    log_researcher,metric_researcher,deployment_researcher])
synthesis = LlmAgent(name="incident_synthesis",model=MODEL,
    instruction="Synthesize evidence. Logs: {logs_report}; metrics: {metrics_report}; "
                "deploys: {deploy_report}. Distinguish observations from hypotheses.",
    output_key="incident_evidence")
planner = LlmAgent(name="incident_remediation_planner",model=MODEL,
    instruction="Based on {incident_evidence}, propose safe options and approval requirements. "
                "DO NOT execute changes or claim deployment, rollback or restart occurred.",
    output_key="remediation_plan")
incident_system = SequentialAgent(name="incident_response_system",sub_agents=[
    triage,investigation_team,synthesis,planner])
print("Capstone agent defined:",incident_system.name)
if RUN_LIVE:
    answer = await run_agent(incident_system,
        f"Investigate this simulated incident: {json.dumps(SIMULATED_INCIDENT)}")
    print(answer["text"])

## 29. Capstone C — a deterministic human approval + verification state machine (offline)

This offline demonstration shows the policy that **must** surround any real-world remediation. The real ADK tool approval equivalent was shown in §24. Here we verify rejection cannot mutate even simulated state.

In [ ]:
MOCK_SERVICE = {"payment_running":True,"restart_count":0}

def simulated_remediation(service:str,approved:bool,approver_role:str) -> dict:
    if service != "payment": return {"status":"blocked","reason":"allowlist"}
    if not approved or approver_role != "oncall_admin":
        return {"status":"blocked","reason":"approval or role missing"}
    MOCK_SERVICE["restart_count"] += 1
    return {"status":"simulated_success","service":service}

def verify_simulated_change(before:int) -> dict:
    return {"changed":MOCK_SERVICE["restart_count"]>before,
            "current_restart_count":MOCK_SERVICE["restart_count"]}

before = MOCK_SERVICE["restart_count"]
assert simulated_remediation("payment",False,"oncall_admin")["status"]=="blocked"
assert verify_simulated_change(before)["changed"] is False
approved = simulated_remediation("payment",True,"oncall_admin")
print(approved, verify_simulated_change(before))

## 30. Streaming outputs and event telemetry

ADK `run_async()` already yields execution events. Use event authors, content, tool calls, usage and final flags for UIs. Token-by-token streaming additionally depends on `RunConfig` streaming settings/model support; don't equate all event streams with token streaming. [Runtime docs](https://adk.dev/runtime/).

In [ ]:
async def watch_events(agent, prompt: str):
    service = InMemorySessionService()
    sid=uuid.uuid4().hex
    await service.create_session(app_name="stream_demo",user_id=USER,session_id=sid)
    runner=Runner(agent=agent,app_name="stream_demo",session_service=service)
    async for ev in runner.run_async(user_id=USER,session_id=sid,
                                    new_message=user_text(prompt)):
        print("event_author=",ev.author,"final=",ev.is_final_response())
        if ev.content and ev.is_final_response():
            print("final:"," ".join(p.text or "" for p in ev.content.parts))
if RUN_LIVE:
    await watch_events(hello_agent,"Describe agent event streaming")

## 31. Artifacts — binary files/images apart from text state

ADK's `ArtifactService` stores versioned binaries, e.g. reports, images and extracted documents. A session state value is NOT the binary content. This example configures a local teaching service; exact artifact lifecycle should follow the [official artifact guide](https://adk.dev/artifacts/).

In [ ]:
from google.adk.artifacts import InMemoryArtifactService

artifact_service = InMemoryArtifactService()
sample_report = types.Part.from_bytes(data=b"incident_id,status\nINC-101,simulated\n",
                                     mime_type="text/csv")
print("Artifact part ready:", sample_report.inline_data.mime_type)
# For a real ADK run, integrate artifact_service into Runner and call
# ToolContext.save_artifact() / load_artifact() in an agent tool.

## 32. Google Search grounding (extra native ADK tool)

`google_search` is a built-in grounding tool for supported Gemini configurations. It is **not** an offline search index and can have model/tool combination restrictions. Verify Google API model/service availability and pricing. [Grounding docs](https://adk.dev/grounding/google_search_grounding/).

In [ ]:
from google.adk.tools import google_search
search_agent = LlmAgent(name="official_docs_researcher",model=MODEL,
    instruction="Search the web for official evidence, name sources, and avoid guesswork.",
    tools=[google_search])
if RUN_LIVE:
    print((await run_agent(search_agent,
      "Find the latest official Google ADK release notes"))["text"])

## 33. Multimodal inputs and code execution

ADK supports multimodal messages via GenAI `Content`/`Part` (images/audio/video depend on model/runtime), and execution tooling through appropriate controlled sandboxes. Do NOT evaluate generated arbitrary code on the notebook host. See [Agent Runtime code execution](https://adk.dev/tools/google-cloud/code-exec-agent-engine/).

In [ ]:
# Local multimodal payload construction ONLY; no API call
minimal_png = bytes.fromhex("89504e470d0a1a0a")  # header only, NOT a complete image
print("PNG header available for explaining multipart messages:",minimal_png.hex())
# Real usage requires a COMPLETE, validated image file:
# from google.genai import types
# image_part = types.Part.from_bytes(data=Path("sample.png").read_bytes(),mime_type="image/png")
# multimodal_message = types.Content(role="user",parts=[types.Part(text="Describe"),image_part])

## 34. MCP tools (Model Context Protocol) — remote/local tool discovery

`McpToolset` integrates MCP servers (stdio, streamable HTTP). Treat arbitrary servers as untrusted and grant the minimum filesystem, network, credential and tool scope. This example is only a **configuration snippet**, not an automatic process launch. [Official MCP documentation](https://adk.dev/tools-custom/mcp-tools/).

In [ ]:
# OPTIONAL integration skeleton — don't run an untrusted executable/server.
# pip install 'google-adk[a2a]' mcp   # extras vary by integration
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

# Set an explicit, trusted URL and auth header before enabling this integration.
# mcp_toolset = McpToolset(connection_params=StreamableHTTPConnectionParams(
#     url=os.environ["TRUSTED_MCP_URL"],
#     headers={"Authorization": f"Bearer {os.environ['TRUSTED_MCP_TOKEN']}"}
# ))
# mcp_agent = LlmAgent(name="trusted_mcp_agent",model=MODEL,
#                      instruction="Use approved tools only.", tools=[mcp_toolset])
print("MCP classes imported; no network connection attempted")

## 35. A2A protocol — agent-to-agent interoperability beyond one process

ADK can **expose** an agent as an A2A service and **consume** remote A2A agents. This enables teams to run independent services with authentication, discovery, task updates and contract-based communication. It is not the same thing as a local `sub_agents` list. Full A2A execution requires an A2A server and appropriate credentials/network configuration.

Official: [A2A introduction](https://adk.dev/a2a/intro/) · [A2A quickstart](https://adk.dev/a2a/quickstart-exposing/).

In [ ]:
A2A_ARCHITECTURE = {
 "coordinator":"incident_coordinator_adk",
 "remote_agents":["compliance_agent_on_team_a","network_agent_on_team_b"],
 "transport":"A2A over authenticated network",
 "controls":["identity","authorization","timeouts","audit","idempotency"]
}
pprint(A2A_ARCHITECTURE)
# Real remote agent samples are version- and deployment-specific: follow the
# official linked A2A quickstart rather than assuming a reachable remote URL.

## 36. Multi-model integrations — Gemini, OpenAI, Claude, Ollama (extra)

ADK is **model-flexible**. Gemini is simplest; Google docs also document OpenAI, LiteLLM, Anthropic, Ollama, vLLM and others. Model choice affects tool calling, `output_schema`, thinking, streaming and latency. Never assume every feature is uniform across providers.

Reference: [Models](https://adk.dev/agents/models/) · [LiteLLM](https://adk.dev/agents/models/litellm/).

In [ ]:
MODEL_CAPABILITY_CHECKLIST = [
 "Tool calling", "Structured output with tools", "Streaming", "Context length",
 "Grounding", "Multimodal", "Safety filters", "Regional data controls",
 "Cost/latency", "SDK integration tests"
]
print("Check before choosing a non-Gemini model:")
for cap in MODEL_CAPABILITY_CHECKLIST: print("-",cap)

## 37. Agent skills, reusable instruction assets, code-first agent projects

Recent ADK builds support skills-related lifecycle capabilities; **some are experimental and must be explicitly enabled**. Skills are not the same as tools or an arbitrary permission system. Verify the installed minor release/feature flag. [Release notes](https://github.com/google/adk-python/releases) · [Skills](https://adk.dev/skills/).

In [ ]:
# Release 2.10 feature highlights include experimental skill lifecycle:
# os.environ["ADK_ENABLE_SKILL_LIFECYCLE"] = "1"  # only after reading feature docs

PROJECT_STRUCTURE = """my_incident_agent/
    __init__.py
    agent.py          # exports root_agent
    .env              # local secret config, gitignored
    tests/
    evals/
"""
print(PROJECT_STRUCTURE)

## 38. Logging, traces, OpenTelemetry, plugins, usage and cost

`adk web` provides a local developer UI showing agent traces; ADK integrates with OpenTelemetry/Google Cloud Trace and observability vendors. Production observability must record model/tool latency, failures, token usage, request IDs and human approvals, while redacting sensitive prompts/tools. Google ADK 2.10 release also describes evaluation metrics for duration, token consumption and model call counts.

Official: [Tracing](https://adk.dev/observability/tracing/) · [Metrics](https://adk.dev/observability/metrics/) · [Plugins](https://adk.dev/plugins/).

In [ ]:
def summarize_demo_metrics(events: list[dict]) -> dict:
    """Pure-Python illustration; not an SDK tracer replacement."""
    durations=[row["duration_ms"] for row in events]
    return {"calls":len(events),"total_duration_ms":sum(durations),
            "max_duration_ms":max(durations,default=0)}
print(summarize_demo_metrics([
 {"duration_ms":31},{"duration_ms":120},{"duration_ms":9}]))

## 39. Evals — offline tests, trajectory/tool validation, LLM judges

Agent evaluations should cover correctness, tool selection, policy compliance, latency/cost, task success, and multi-agent coordination. Google ADK has an `adk eval` CLI and official evaluation framework with criteria and custom metrics. Don't rely on answer similarity alone.

Official: [Evaluation](https://adk.dev/evaluate/) · [Criteria](https://adk.dev/evaluate/criteria/).

In [ ]:
def evaluate_mock_responders():
    assert get_service_health("payment")["metrics"]["healthy"] is False
    assert get_service_health("unknown")["found"] is False
    assert restrict_service_tool(None,{"service":"unapproved"},None)["error"]
    assert classify_mock_incident("severe outage").severity == "critical"
    assert simulated_remediation("unknown",True,"oncall_admin")["status"]=="blocked"
    return {"passed":5,"type":"deterministic offline contract tests"}
print(evaluate_mock_responders())

# CLI examples (terminal, not executed):
# adk eval path/to/agent/ path/to/evalset.json
# See documentation for exact evaluation dataset/criteria schema on your ADK version.

## 40. Packaging + ADK CLI + Web UI + deployment

From notebook to service: export a `root_agent` from a Python module, keep config in environment, and start locally with:

```bash
adk create my_agent
adk run my_agent
adk web --port 8000            # local DEVELOPMENT UI only
adk api_server --port 8000     # consult authentication and exposure guidance before production
```

Deployment choices: Cloud Run, Agent Runtime/Agent Engine, GKE, containers, or your own orchestrated service. Use cloud IAM, secrets, request authentication, rate limits, persistent session backend, distributed tracing, rollback and eval gates. `adk web` should **not be treated as a production dashboard/server**.

Docs: [Quickstart](https://adk.dev/get-started/python/) · [Deploy](https://adk.dev/deploy/) · [agents-cli](https://google.github.io/agents-cli/).

In [ ]:
# Export a minimal app module you can inspect or copy to a separate package.
# Does NOT start a web server and is safe to write in a teaching folder.
agent_module_text = '''from google.adk.agents import LlmAgent
root_agent = LlmAgent(name="deployed_demo", model="gemini-2.5-flash",
                      instruction="You are a clear and careful assistant.")
'''
module_dir=Path("google_adk_course_project")
module_dir.mkdir(exist_ok=True)
(module_dir/"__init__.py").write_text("from . import agent\n",encoding="utf-8")
(module_dir/"agent.py").write_text(agent_module_text,encoding="utf-8")
print("Created:",(module_dir/"agent.py").resolve())

## 41. Production engineering checklist — important for GCC/MNC use

- **Identity and auth:** API gateway, signed identities, trusted tenants, RBAC and per-tool authorization.
- **Tools:** non-destructive defaults, approval on mutations, idempotency keys, rate limits, parameter validation.
- **Data:** minimum privilege, prompt/tool log redaction, data retention, access-controlled memory/artifacts.
- **Orchestration:** graph exit conditions, bounded loops, timeouts, retries, dead-letter/compensation.
- **Quality:** test fixtures, eval datasets, regression gates, human review and feedback loop.
- **Observability:** spans, tokens, tool traces, per-agent latency, cost and incident correlation IDs.
- **Ops:** staging → canary → production, versioned model prompts, fallback policies, explicit owner/on-call.
- **MCP/A2A:** mutually authenticated endpoints, allowlists, schema validation, SSRF safeguards and trust boundaries.

**Crucial:** Production workflows should not infer permission to restart, deploy, delete, purchase or contact people solely from a generative model response.

## 42. Student assignments — progressive exercises

1. Make a ReAct-like tool agent with 3 read-only tools; examine event traces.
2. Create a Pydantic classifier and save output to `output_key`.
3. Build 2-turn stateful conversation using `InMemorySessionService`, then SQLite persistence.
4. Build memory ingestion/retrieval with separate `MemoryService`.
5. Create a hierarchical manager + two specialist agents; contrast `AgentTool` vs `sub_agents`.
6. Implement 3-stage SequentialAgent and independent two-branch ParallelAgent.
7. Build a bounded quality LoopAgent and test termination.
8. Create native Workflow with `Event(route=...)` and `JoinNode`.
9. Put a human approval `RequestInput` node before simulated remediation.
10. Implement production-style evals and tracing; package your `root_agent` for a local `adk web` run.
11. Capstone: replace mock metrics/tools with an authorized test API and compare latency/errors vs your OpenAI Agents SDK notebook.

## 43. Official sources and version caveats

All references are from Google/ADK official documentation or the Google ADK official GitHub repository as viewed **10 October 2026**.

| Subject | Official link |
|---|---|
| Source code and release notes | [google/adk-python](https://github.com/google/adk-python) · [Releases](https://github.com/google/adk-python/releases) |
| Installation | [Python getting started](https://adk.dev/get-started/python/) |
| LlmAgent and structured output | [Simple agents](https://adk.dev/agents/llm-agents/) |
| Tool creation, callbacks, confirmations | [Function tools](https://adk.dev/tools-custom/function-tools/) · [Callbacks](https://adk.dev/callbacks/) · [Action confirmations](https://adk.dev/tools-custom/confirmation/) |
| Workflows 1.x | [Sequential](https://adk.dev/agents/workflow-agents/sequential-agents/) · [Parallel](https://adk.dev/agents/workflow-agents/parallel-agents/) · [Loop](https://adk.dev/agents/workflow-agents/loop-agents/) |
| Native ADK 2.x graphs | [Graph routes](https://adk.dev/graphs/routes/) · [Data](https://adk.dev/graphs/data-handling/) · [Human input](https://adk.dev/graphs/human-input/) |
| Sessions/memory | [Sessions](https://adk.dev/sessions/session/) · [Memory](https://adk.dev/sessions/memory/) |
| Resume/rewind | [Resume agents](https://adk.dev/runtime/resume/) · [Rewind](https://adk.dev/sessions/session/rewind/) |
| MCP/A2A | [MCP tools](https://adk.dev/tools-custom/mcp-tools/) · [A2A introduction](https://adk.dev/a2a/intro/) |
| Artifacts, grounding, evaluations | [Artifacts](https://adk.dev/artifacts/) · [Grounding](https://adk.dev/grounding/google_search_grounding/) · [Evaluation](https://adk.dev/evaluate/) |
| Local development/deployment | [Quickstart](https://adk.dev/get-started/python/) · [ADK CLI](https://adk.dev/runtime/) · [Cloud Run](https://adk.dev/deploy/cloud-run/) |

### Known limitations / honesty checklist

- Local environment for notebook generation had **no network package installation**, so the ADK import/runtime examples were **not run end-to-end**. Python syntax and notebook metadata validation were performed offline.
- `google-adk==2.10.0` is a specific 2026 version; later versions may modify APIs.
- Model access varies by account; structured outputs with tools are **model-dependent**.
- Graph-based `RequestInput` requires a runtime/client capable of collecting and supplying replies. The notebook demonstrates correct construction, not a working chat frontend.
- `DatabaseSessionService` stores sessions; it doesn't promise identical LangGraph graph snapshot/time-travel semantics.
- Native 2.x graph API is newer; treat every unexecuted graph demo as **research-backed sample code** and integration-test it before teaching/production.